# 03 — Absolute Sustainability Ratio

    ASR = emissions / allocated carrying capacity

The numerator is notebook 02's emissions table. The denominator is each
country's fair share of the global carbon budget, which pyaesa derives from
the IPCC AR6 pathways under **equal per capita** allocation — every person
alive gets the same share.

- **ASR < 1** — living within a fair share
- **ASR > 1** — exceeding it, by that multiple

**Outputs**
- `data_viz/asr.json` — `{iso3: {year: asr}}` for the D3 app
- `data_viz/asr.csv` — long form, both budget bounds

In [ ]:
import json

import pandas as pd
import pyaesa

from config import (
    ASR_FILE, LCA_FILE, LCA_VERSION, LCIA_METHOD, PROJECT, ROOT, VIZ,
    YEARS, YEAR_COLS,
)

pyaesa.set_workspace(top_path=str(ROOT))

## 1. Compute

`r_f` is pinned to the countries actually present in the emissions table.
Without it pyaesa allocates a budget to every World Bank country and then
fails on the ones with no emissions data to divide.

pyaesa runs the whole chain — allocation, then budget, then ratio — so there
is no separate step to call first. Expect a few minutes.

In [ ]:
countries = sorted(pd.read_csv(LCA_FILE)["r_f"].unique())
print(f"Computing ASR for {len(countries)} countries...")

result = pyaesa.deterministic_asr(
    project_name=PROJECT,
    source="iso3",
    fu_code="L1.a",
    years=list(YEARS),
    lcia_method=LCIA_METHOD,
    r_f=countries,
    base_asocc_args={"include_lcia_based_allocation_methods": False},
    lca_args={"external_lca": {"active": True, "version_name": LCA_VERSION}},
)

## 2. Inspect

`cc_bound` carries the budget's uncertainty: `min_cc` is the conservative
budget and so the higher ratio, `max_cc` the generous one. Everything below
reports `min_cc`.

Palau and New Caledonia sit far above their neighbours. Both are territorial
emissions divided by a small resident population — Palau hosts several times
its own population in visitors each year, New Caledonia runs nickel smelters.
Worth flagging in the visualisation rather than presenting flat.

In [ ]:
asr = pd.read_csv(ASR_FILE)

long = (
    asr.melt(
        id_vars=["r_f", "cc_bound"], value_vars=YEAR_COLS,
        var_name="year", value_name="asr",
    )
    .astype({"year": int})
    .rename(columns={"r_f": "iso_code"})
)

snapshot = (
    long.query("cc_bound == 'min_cc' and year == 2020")
    .set_index("iso_code")["asr"]
    .sort_values()
)
print(f"{len(snapshot)} countries in 2020\n")
print("Lowest 10:\n", snapshot.head(10).round(3), "\n")
print("Highest 10:\n", snapshot.tail(10).round(1))

## 3. Export

In [ ]:
VIZ.mkdir(exist_ok=True)
long.to_csv(VIZ / "asr.csv", index=False)

export = {
    iso: dict(zip(g["year"], g["asr"].round(4)))
    for iso, g in long.query("cc_bound == 'min_cc'").groupby("iso_code")
}
(VIZ / "asr.json").write_text(json.dumps(export, indent=2))

print(f"{len(export)} countries -> data_viz/asr.json, data_viz/asr.csv")